# 10 · Hipótesis de la literatura: hora de la lluvia, mar absoluto, índices E y C, lluvia satelital

> **No es un sistema de alerta ni una predicción para decidir evacuaciones.** Ante una emergencia llama al 115 (INDECI).

Las cuatro hipótesis salen del capítulo `docs/09-literatura-mecanismos.md` y se fijaron **antes** de mirar resultados:

| | Hipótesis | Variable | Fuente |
|---|---|---|---|
| T23 | Los huaicos vienen con lluvia de la tarde y corta e intensa | hora del pico, fracción de la lluvia entre 12 y 20 h, horas con lluvia, racha más larga, fracción en 3 h | ERA5 horario (Open-Meteo), Chosica y cuenca alta |
| T24 | La convección responde a la TSM **absoluta** (umbral), no solo a la anomalía | grados-día sobre 25, 26, 27 y 28 °C en 7, 14 y 30 días y racha de días sobre cada umbral | NOAA OISST (Niño 1+2, costa de Perú, costa de Lima) |
| T25 | Los índices E (costero) y C (central) explican la frecuencia mejor que el ICEN o el ONI | E, C, ICEN y ONI de enero a abril; E y C del mes previo en el modelo diario | IGP (E, C, ICEN), NOAA (ONI) |
| T28 | La lluvia satelital (CHIRPS v3, tiempo de IMERG) mejora o complementa la de ERA5 | lluvia de 1 y 3 días y de los 7 días previos en 5 cajas de 0,2° | Climate Hazards Center |

Criterios, iguales al capítulo 05: dejar fuera una temporada cada vez; cambio de la precisión media (AP) contra el mejor modelo del capítulo 05 (lluvia + condiciones de ayer), con intervalo por bootstrap de temporadas; nula en que el bloque nuevo se desplaza cíclicamente dentro de cada temporada.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from illapa_research import estudio_eventos as ee, evaluacion_diaria as E, filtro, hipotesis_literatura as H
from illapa_research import modelo_diario as MD, nino as N, predictores as P, quebradas as q
from illapa_research.catalogo import CATALOGO, cargar_cuencas, cargar_todos

warnings.filterwarnings("ignore")
RAIZ = Path("..") if Path("../datos").exists() else Path(".")
D, X, y, ts, mes, folds = H.contexto_modelo(RAIZ)
base = X["lluvia+antecedentes"]
yv = y.to_numpy()
rb = MD.validar_anidado(base, y, ts, folds, n_jobs=8)
print(D.inventario["puntos_era5"], D.inventario["fuente_era5"])
print("modelo base:", {k: round(v, 3) for k, v in MD.metricas_completas(yv, rb.p, E.p_climatologia(y, mes, folds)).items() if k in ("ap", "ap_base", "auc", "aciertos_con_5pc_alarmas")})

In [ ]:
def bloque(nombre, extra, n_nula=8):
    r = H.correr_bloque(base, extra, y, ts, folds)
    d = H.ap_pareado(yv, r.p, rb.p, ts)
    m = MD.metricas_completas(yv, r.p, rb.p)
    nula = H.nula_bloque(base, extra, y, ts, folds, n=n_nula)
    return {"bloque": nombre, "columnas": extra.shape[1], "ap": m["ap"], "delta_ap": d["delta_ap"], "p5": d["p5"], "p95": d["p95"],
            "prob_mejora": d["prob_mejora"], "nula_p95": float(np.percentile(nula, 95)),
            "aciertos_5pc": m["aciertos_con_5pc_alarmas"]}

resumen = []

## T23 · Hora de la lluvia

Se compara, entre los días lluviosos (lluvia diaria de ERA5 de al menos 1 mm), los días con huaico y los demás. AUC = 0,5 quiere decir sin diferencia. La última columna repite el AUC solo entre los días de lluvia fuerte (percentil 90), para que no sea solo intensidad.

In [ ]:
hor = ee.cargar_horario(RAIZ / "datos" / "open_meteo_horario_estudio.json")
T = H.variables_horarias(hor, D.F.index)
Tm = T[D.inventario["mask"]]
for punto in ("chosica", "cuenca-alta"):
    print(punto)
    display(H.comparar_dias_lluviosos(Tm, y, ts, np.ones(len(y), bool), punto, n_boot=200).round(3))

In [ ]:
temporales = [c for c in Tm.columns if not c.endswith(("_lluvia_dia", "_lluvia_tarde"))]
print(H.cribado_estratificado(Tm[temporales], y, mes, ts, n_boot=200).round(3).to_string())

¿Mejoran el modelo? Las variables del propio día son un *nowcast* (lag 0); las de ayer sí son antecedentes.

In [ ]:
duracion = ["chosica_horas_humedas", "cuenca_alta_horas_humedas", "chosica_racha_max_h", "cuenca_alta_racha_max_h", "chosica_frac_3h_max", "cuenca_alta_frac_3h_max"]
ayer = T[temporales].shift(1).add_suffix("_ayer")[D.inventario["mask"]]
resumen.append(bloque("T23 timing del día, 10 columnas (nowcast)", Tm[temporales]))
resumen.append(bloque("T23 duración y concentración del día, 6 columnas (nowcast)", Tm[duracion]))
resumen.append(bloque("T23 timing de ayer, 10 columnas (antecedente)", ayer))
pd.DataFrame(resumen).round(3)

## T24 · Temperatura absoluta del mar

In [ ]:
sst = P.cargar_sst(RAIZ / "datos" / "enso" / "oisst_diario.csv")
sst = sst[~sst.index.duplicated(keep="last")]
S = H.variables_sst_absoluta(sst, D.F.index)
Sm = S[D.inventario["mask"]]
print({c: int((Sm[c] > 0).sum()) for c in Sm.columns if "racha" in c})
print(sst.describe().round(1))

In [ ]:
# AUC calculada dentro de cada mes de calendario: quita el ciclo estacional (el mar se calienta de diciembre a marzo)
c = H.cribado_estratificado(Sm, y, mes, ts, n_boot=200)
print(c.head(12).round(3).to_string())
print("variables que separan:", int(c.separa.sum()), "de", len(c))

In [ ]:
niveles = [c for c in Sm.columns if c.endswith("_nivel")]
umbral_alto = [c for c in Sm.columns if any(k in c for k in ("gd26", "gd27", "gd28", "racha_26", "racha_27", "racha_28")) and "costa_lima" not in c]
resumen.append(bloque("T24 TSM absoluta, nivel (3 cajas)", Sm[niveles]))
resumen.append(bloque("T24 grados-día y rachas sobre 26-28 °C (Niño 1+2, costa de Perú)", Sm[umbral_alto]))
resumen.append(bloque("T24 todo (51 columnas)", Sm))
pd.DataFrame(resumen[-3:]).round(3)

## T25 · Índices E y C

Contraste por temporada (una cifra por temporada, como en el capítulo 03): correlación de Spearman de cada índice (enero a abril) con el número de quebradas ubicadas activas y con los días con huaico, con valor p por permutación; y deviance de Poisson dejando una temporada fuera.

In [ ]:
ec = H.cargar_ec(RAIZ / "datos" / "enso" / "igp_ec_ersstv5.txt")
icen = N.cargar_icen(RAIZ / "datos" / "enso" / "igp_icen.txt")
oni = N.cargar_oni(RAIZ / "datos" / "enso" / "noaa_oni.txt")
ind = H.ec_por_temporada(ec, icen, oni)
ind = ind[ind.index >= 1984]
cuencas = filtro.cuencas_a_tabla(cargar_cuencas())
refs = q.cargar_referencias(CATALOGO / "quebradas_referencias.csv")
eventos = cargar_todos(solo_dia_exacto=True, sin_baja_confianza=True)
hist = filtro.asignar_historial(eventos, refs, cuencas)
nq = hist.activaciones.drop_duplicates(["fecha", "slug"]).groupby("temporada").slug.nunique()
dias = pd.Series([N.temporada_de(f) for f in sorted({e.fecha for e in eventos})]).value_counts()
print(ind.corr().round(2))
for nombre, res in (("quebradas ubicadas activas", nq), ("días con huaico", dias)):
    print(nombre)
    display(H.correlacion_con_temporadas(ind, res, n_perm=2000).round(3))

In [ ]:
conj = {"media": [], "ICEN": ["ICEN"], "ONI": ["ONI"], "E": ["E"], "C": ["C"], "E+C": ["E", "C"], "ICEN+C": ["ICEN", "C"], "ICEN+ONI": ["ICEN", "ONI"], "E+C+ICEN+ONI": ["E", "C", "ICEN", "ONI"]}
pd.concat({"quebradas activas": H.validacion_poisson(ind, nq, conj).set_index("modelo")["deviance_dejando_una_fuera"],
           "días con huaico": H.validacion_poisson(ind, dias, conj).set_index("modelo")["deviance_dejando_una_fuera"]}, axis=1).round(1)

In [ ]:
V = H.variables_ec_mensuales(ec, D.F.index)[D.inventario["mask"]]
resumen.append(bloque("T25 E y C del mes previo y su cambio (4 columnas)", V))
pd.DataFrame(resumen[-1:]).round(3)

## T28 · Lluvia satelital (CHIRPS v3 con tiempo de IMERG)

CHIRPS diario reparte el total de cada pentada (estaciones + satélite) entre los días con el tiempo de IMERG; por eso el *reparto* entre días es independiente de ERA5, pero el total de la pentada no es una medida diaria. Cajas fijadas antes de mirar: los puntos de ERA5 ± 0,1°.

In [ ]:
ruta = RAIZ / "datos" / "chirps_v3_sat_diario.csv"
if ruta.exists():
    ch = H.cargar_chirps(ruta)
    print("CHIRPS:", ch.index.min().date(), "a", ch.index.max().date(), list(ch.columns))
    corr = {"chosica": "chosica_h_lluvia_1d", "santa-eulalia-alta": "sea_lluvia_1d", "matucana": "mat_lluvia_1d", "ticlio": "tic_lluvia_1d", "selva-tarma": "selva_lluvia_3d"}
    display(H.comparar_con_era5(ch, D.F, {k: v for k, v in corr.items() if k != "selva-tarma"}).round(3))
else:
    print("Falta datos/chirps_v3_sat_diario.csv: el servidor del Climate Hazards Center no respondió. Ver docs/10.")

In [ ]:
if ruta.exists():
    Cx = H.variables_chirps(ch, D.F.index)[D.inventario["mask"]]
    print("días con CHIRPS:", int(Cx.notna().any(axis=1).sum()), "desde", Cx.dropna(how="all").index.min().date())
    # solo se compara en las temporadas con CHIRPS (1998 en adelante), con los mismos pliegues para los tres modelos
    display(H.evaluar_chirps(base, Cx, y, ts).round(3))


### T28b · Sin CHIRPS: PERSIANN-CDR y el análisis de pluviómetros del CPC

El servidor del Climate Hazards Center no responde, así que T28 se prueba con dos series públicas de NOAA, también independientes de ERA5 (`src/illapa_research/lluvia_satelital.py`, `descargar-datos --solo persiann-cdr cpc-lluvia`):

- **PERSIANN-CDR** (NCEI, 0,25°): lluvia satelital infrarroja ajustada a totales mensuales de pluviómetros (GPCP). Noviembre a abril desde la temporada 1984.
- **CPC Global Unified Gauge-Based Analysis** (PSL, 0,5°): solo pluviómetros interpolados, sin satélite ni reanálisis.

Se lee la celda que contiene el centro de cada punto (los centros de las cajas de arriba), fijados antes de mirar. Las pruebas son las mismas: acuerdo con ERA5, lluvia en los días con huaico, y añadir o sustituir las columnas de lluvia en el modelo `lluvia+antecedentes` (pliegues de una temporada fuera, afinado anidado), más una versión de GPU con regularización fija y 500 mundos nulos en que el bloque se desplaza dentro de cada temporada.


In [ ]:

from illapa_research import lluvia_satelital as LS

if (RAIZ / "datos" / "lluvia_sat").exists():
    T28 = LS.analizar(RAIZ, n_nulas=500)
    for nombre, tabla in T28.items():
        print("\n##", nombre)
        display(tabla.round(3))
else:
    print("Faltan datos/lluvia_sat/: ejecuta `descargar-datos --solo persiann-cdr cpc-lluvia`.")


## Resumen

In [ ]:
pd.DataFrame(resumen).round(3)